# 码流图像分类

**目标：识别手写数字 0–9，自行设置训练轮数和 batch size 完成微调、比较验证结果，并分析一个错例。**

课程仓库：https://github.com/Franklin-L/byteformer-mnist-course

使用 ImageNet 预训练 ByteFormer Tiny，将分类头调整为 10 类，在 MNIST 上进行全参数微调。模型输入为图像编码后的 JPEG 文件字节。

自行设置训练参数，运行后查看自己的曲线与指标，解释参数变化的影响。

按顺序运行单元格，训练参数与结果将保存在 outputs 目录中。

## 开始之前

1. 在 Kaggle 新建 Notebook，并导入本文件。
2. 在设置中开启 **GPU** 和 **Internet**，如提示账号验证，先按平台提示完成。
3. **从上到下逐格运行，不要第一次直接 Run All。** 等当前单元格完成后再继续。
4. 代码位于可写的 `/kaggle/working/`。会话结束前下载最后生成的作业 ZIP。

GPU 是否可用及额度以你的 Kaggle 账号为准；无可用 GPU 时，参照 README 的 AutoDL 路线。

## 1. 检查 GPU

点击下一格左侧的运行按钮。看到 `GPU available: True` 才继续主线。

In [ ]:
import sys
import subprocess
# 在子进程检查，避免安装依赖前把旧 NumPy 载入笔记本内核。
gpu_check = """
import sys, torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "主线需要 GPU。请在 Kaggle 设置中开启 GPU；无可用 GPU 时参照 README 的 AutoDL 或 CPU 路线。"
"""
subprocess.run([sys.executable, "-c", gpu_check], check=True)


## 2. 获取代码并安装轻量依赖

`!` 表示在 Python 笔记本中执行终端命令；`%cd` 用来切换工作目录。这里保留 Kaggle 预装 PyTorch，不重新安装 CUDA 或指定旧版本 PyTorch。

In [ ]:
from pathlib import Path
course_dir = Path("/kaggle/working/byteformer-mnist-course")
if not course_dir.exists():
    !git clone https://github.com/Franklin-L/byteformer-mnist-course.git /kaggle/working/byteformer-mnist-course
else:
    print("已存在课程目录，将复用；已有实验结果不会自动删除。")
assert (course_dir / "train.py").is_file(), "代码获取失败，请检查 Internet 设置或按 README 上传 ZIP。"
%cd /kaggle/working/byteformer-mnist-course
!python -m pip install -r requirements.txt

## 3. 准备数据和官方权重

仓库附带 4 个 MNIST 原始压缩文件。脚本会校验数据，并下载约 61 MB 官方预训练权重。已存在且通过校验的文件会复用。不要跳过下载或校验错误。

In [ ]:
!python prepare.py

## 4. 自行设置训练轮数与 batch size

- `EPOCHS`：自行输入正整数，表示本次计划学习训练集多少遍；不固定轮数。
- 训练/验证/测试分别使用 **50,000 / 1,000 / 10,000** 个样本，默认种子 **42**。
- `batch-size`：每批处理的样本数，由你设置，可参考 32。
- `lr=0.0001`：主干学习率；新的分类头使用其 10 倍。
- `lr-milestones=4 6`、`lr-gamma=0.2`：第 4、6 轮结束后学习率乘以 0.2，脚本自动执行。
- `best.pt` 由**验证集**选出，最后才用测试集评估。

官方 60,000 张训练图固定分为 50,000 张训练池和 10,000 张验证池；本实验使用全部训练池和验证池前 1,000 张，**其余 9,000 张验证池样本保留未用**。官方 10,000 张测试图独立保留，只在训练结束后评估。实际验证样本数是 1,000，不是 10,000。轮数根据学习目标与验证表现自行选择；batch size可参考32；如显存不足，可选择更小的正整数，并记录实际设置。

**保护结果：** 如果 `outputs/baseline` 已有结果，脚本会报错。需要重跑时，请改用新输出目录，并同步修改下方所有 checkpoint 和结果路径，不要随意删除自己已有的记录。

In [ ]:
EPOCHS = int(input("请输入本次训练轮数（正整数）："))
assert EPOCHS > 0, "训练轮数必须是正整数。"
BATCH_SIZE = int(input("请输入 batch size（参考32，正整数）："))
assert BATCH_SIZE > 0, "batch size 必须是正整数。"
!python train.py --epochs {EPOCHS} --train-samples 50000 --val-samples 1000 --test-samples 10000 --batch-size {BATCH_SIZE} --lr 0.0001 --lr-milestones 4 6 --lr-gamma 0.2 --output outputs/baseline

## 5. 读出指标与学习曲线

Loss 是训练误差，通常希望下降；accuracy 是预测正确的比例，通常希望上升。JSON 中准确率为 0–1，下面的代码会转换为百分比。

请在报告里回答：训练损失是否总体下降？验证准确率是否提高？验证结果和测试结果能否混写？

In [ ]:
import json
from IPython.display import display, Image as DisplayImage
baseline_dir = Path("outputs/baseline")
assert (baseline_dir / "metrics.json").is_file(), "训练尚未成功完成，请先检查上一个单元格的错误。"
baseline = json.loads((baseline_dir / "metrics.json").read_text())
print("最佳验证轮次:", baseline["best_epoch"])
print(f"最佳验证准确率: {baseline['best_validation_accuracy']:.2%}")
print(f"最终测试准确率: {baseline['test']['accuracy']:.2%} ({baseline['test']['correct']}/{baseline['test']['samples']})")
print(f"脚本记录耗时: {baseline['elapsed_seconds']:.2f} 秒（不含安装下载）")
for name in ["curves.png", "predictions.png", "confusion_matrix.png"]:
    print(name)
    display(DisplayImage(filename=str(baseline_dir / name)))

预测图显示前 12 个测试样本，并额外加入最多 4 个实际错例，便于课堂分析。整体测试准确率见上方指标。 混淆矩阵的行是真实数字，列是预测数字；对角线外表示错误。

## 6. 不训练，直接评估与预测

下面从 `best.pt` 读取已经训练好的模型，重复评估全部 10,000 张官方测试图。新结果存为 `evaluation.json`，不会改写训练的 `metrics.json`。

In [ ]:
!python evaluate.py --checkpoint outputs/baseline/best.pt --output outputs/baseline

查看官方测试集第 0 张图。你也可以把 `--index 0` 改成 0–9999 之间的其他索引。

In [ ]:
!python predict.py --checkpoint outputs/baseline/best.pt --index 0

display(DisplayImage(filename="outputs/baseline/prediction_single.png"))

## 7. 自选参数作对比

训练轮数和 batch size 都由自己输入，不规定必须使用哪些数值。建议一次只改变其中一项，另一项沿用基线，便于判断效果；如果两项都变了，在报告中说明比较的局限。

数据划分、随机种子、学习率策略和官方预训练来源保持相同。另存到 `outputs/comparison`；基于训练与验证曲线分析变化，不根据测试成绩反复选参数。


In [ ]:
COMPARISON_EPOCHS = int(input("请输入对比实验的训练轮数（正整数）："))
assert COMPARISON_EPOCHS > 0, "训练轮数必须是正整数。"
COMPARISON_BATCH_SIZE = int(input(f"请输入对比 batch size（基线使用{baseline['config']['batch_size']}）："))
assert COMPARISON_BATCH_SIZE > 0, "batch size 必须是正整数。"
!python train.py --epochs {COMPARISON_EPOCHS} --train-samples 50000 --val-samples 1000 --test-samples 10000 --batch-size {COMPARISON_BATCH_SIZE} --lr 0.0001 --lr-milestones 4 6 --lr-gamma 0.2 --output outputs/comparison

### 对比两次实验

将指标和曲线填入报告，比较验证结果、最佳轮次和耗时，分析参数调整的影响。

In [ ]:
comparison_dir = Path("outputs/comparison")
assert (comparison_dir / "metrics.json").is_file(), "请先完成自选对比实验。"
comparison = json.loads((comparison_dir / "metrics.json").read_text())
for label, result in [("基线实验", baseline), ("自选对比", comparison)]:
    print(f"{label}: 设置轮数={result['config']['epochs']}, batch size={result['config']['batch_size']}, 最佳轮次={result['best_epoch']}, "
          f"验证准确率={result['best_validation_accuracy']:.2%}, "
          f"最终测试准确率={result['test']['accuracy']:.2%}, "
          f"耗时={result['elapsed_seconds']:.2f} 秒")
display(DisplayImage(filename="outputs/comparison/curves.png"))

## 8. 错例分析

读取基线测试输出，获取错误样本的官方测试索引，再次运行预测。

记录样本标签和预测标签，结合笔画特征分析可能的错误原因。

In [ ]:
import numpy as np
with np.load("outputs/baseline/test_predictions.npz") as saved:
    mistake_rows = np.flatnonzero(saved["predictions"] != saved["labels"])
    if len(mistake_rows):
        row = int(mistake_rows[0])
        mistake_index = int(saved["indices"][row])
        print("完整测试集实际错例数:", len(mistake_rows))
        print("官方测试索引:", mistake_index,
              "真实标签:", int(saved["labels"][row]),
              "预测标签:", int(saved["predictions"][row]))
    else:
        mistake_index = None
        print("本次完整测试集没有错例；请真实记录，并分析一个容易混淆的样本。")
if mistake_index is not None:
    !python predict.py --checkpoint outputs/baseline/best.pt --index {mistake_index} --output outputs/baseline/mistake
    display(DisplayImage(filename="outputs/baseline/mistake/prediction_single.png"))

## 9. 可选体验：自己的数字图片

先用仓库附带示例体验。你也可以将图片上传到课程目录，然后把 `assets/example_digit.png` 换成实际图片路径。建议黑底白字、一个居中的手写数字；脚本不会自动反色。

自制图片可能与 MNIST 分布不同。它是额外体验，不替代规定测试集。Softmax 置信度也不是保证预测正确的概率。

In [ ]:
!python predict.py --checkpoint outputs/baseline/best.pt --image assets/example_digit.png --output outputs/custom

display(DisplayImage(filename="outputs/custom/prediction_single.png"))

## 10. 下载作业结果

下一格把 `outputs/` 下的小型结果文件打包成 `/kaggle/working/byteformer_mnist_results.zip`，包括 JSON、CSV、图片和预测数组，**不包含大型 `best.pt`**。

运行后，在 Kaggle 的文件/输出面板找到 ZIP 并下载；也可以尝试单元格显示的下载链接。若要保留完整会话，使用页面提供的保存版本功能；结束会话前先确认材料已下载到自己的电脑。以后继续推理需要另存 `best.pt`。

从仓库下载可编辑的 `docs/学生实验报告模板.docx`，填写报告并与 ZIP 一起提交；也提供 `docs/student_report_template.md`。报告要写自己的参数、指标、自选设置的比较和错例分析。

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink
archive = Path("/kaggle/working/byteformer_mnist_results.zip")
result_files = [p for p in Path("outputs").rglob("*")
                if p.is_file() and p.suffix in {".json", ".csv", ".png", ".npz"}]
assert result_files, "没有找到结果文件，请先完成训练。"
with ZipFile(archive, "w", ZIP_DEFLATED) as bundle:
    for path in result_files:
        bundle.write(path, path.as_posix())
print("已打包:", archive)
print("文件数:", len(result_files))
print("请在 Kaggle 文件/输出面板下载 byteformer_mnist_results.zip。")
display(FileLink("../byteformer_mnist_results.zip"))

## 提交前检查

- 已完成基线与参数对比实验。
- 我知道训练集更新参数、验证集选最佳模型、测试集作最终评估。
- 已记录样本数、轮数、学习率、batch size 和运行环境。
- 已比较验证结果，并完成错例分析。
- 我已下载 ZIP，并填写报告。未完成的部分有明确说明。

CPU 快速路线、AutoDL 操作、网络失败的 ZIP 备用获取方式，以及详细故障处理，请见课程 README。